In [1]:
import pandas as pd

## Import

In [10]:
account = pd.read_csv("../../../data/raw/accounts.csv")
churn_events = pd.read_csv("../../../data/raw/churn_events.csv")
feature_usage = pd.read_csv("../../../data/raw/feature_usage.csv")
subscriptions = pd.read_csv("../../../data/raw/subscriptions.csv")
support_tickets = pd.read_csv("../../../data/raw/support_tickets.csv")

## Load

In [21]:
churn_events.head(10)
# account.head(10)
# feature_usage.head(10)
# subscriptions.head(10)
# support_tickets.head(10)

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive
5,C-a8ac26,A-b20d99,2024-03-04,features,0.00,False,False,False,missing features
6,C-2e2966,A-526d93,2024-07-10,features,34.40,True,False,False,NaN
7,C-ee531b,A-751bd4,2024-09-22,budget,0.00,False,False,False,NaN
8,C-e6a0f0,A-ccabf0,2024-10-31,features,0.00,False,False,False,too expensive
9,C-9328ba,A-0a62f5,2024-10-10,features,0.00,False,False,False,switched to competitor


# Profile

In [ ]:
# churn_events.info()
# account.info()
# feature_usage.info()
# subscriptions.info()
# support_tickets.info()

In [24]:
# Does any account have more than one subscription row?
dupe_check = subscriptions.groupby("account_id").size()
dupe_check.head(10)

account_id
A-00bed1    10
A-00cac8     9
A-0158bb     6
A-016043    11
A-019782     9
A-029f69    16
A-02cd81    14
A-02fac6    11
A-034368     5
A-0354fe     7
dtype: int64

In [25]:
sample = subscriptions[subscriptions["account_id"] == "A-016043"].sort_values("start_date")
print(sample[["subscription_id", "start_date", "end_date", "plan_tier", "mrr_amount", "upgrade_flag", "downgrade_flag", "churn_flag"]])

     subscription_id  start_date    end_date   plan_tier  mrr_amount  \
2282        S-0e1d53  2024-08-02         NaN       Basic         114   
4750        S-d54349  2024-09-11         NaN  Enterprise        3980   
3618        S-769f70  2024-09-28         NaN       Basic         247   
3762        S-0a8cff  2024-09-28         NaN  Enterprise           0   
3676        S-2c52b6  2024-10-09         NaN  Enterprise        2786   
1075        S-6bfd97  2024-10-16         NaN  Enterprise           0   
1509        S-3f0fb6  2024-10-20         NaN         Pro        1323   
4536        S-0abd10  2024-11-23  2024-12-21  Enterprise        4776   
1415        S-d114e2  2024-11-27         NaN       Basic         209   
326         S-463525  2024-12-02         NaN  Enterprise        2587   
2882        S-8985d3  2024-12-02         NaN       Basic         494   

      upgrade_flag  downgrade_flag  churn_flag  
2282         False           False       False  
4750         False           False   

In [26]:
# Does upgrade_flag correlate with mrr_amount = 0 broadly, or was this one account a fluke?
subscriptions.groupby("upgrade_flag")["mrr_amount"].describe()

# Are there other accounts where an upgrade_flag=True row's start_date lines up
# with another row's end_date for the same account (i.e., a real handoff)?
subs_sorted = subscriptions.sort_values(["account_id", "start_date"])
subs_sorted["prev_end_date"] = subs_sorted.groupby("account_id")["end_date"].shift()
handoffs = subs_sorted[subs_sorted["start_date"] == subs_sorted["prev_end_date"]]
print(len(handoffs))

7
